In [1]:
# XGBoost

# Importing the libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

from scipy.optimize import differential_evolution


In [3]:
# Importing the dataset
from datasets import load_dataset

dataset = load_dataset("maharshipandya/spotify-tracks-dataset")
data = dataset["train"].to_pandas()

C:\Users\anton\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\anton\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\anton\.cache\huggingface\hub\datasets--maharshipandya--spotify-tracks-dataset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.

In [5]:
#Select the input factors and target
feature_columns = [6, 7, 8, 9, 11, 13, 14, 15, 16, 17, 18]
target_column = 5
X = data.iloc[:, feature_columns].copy()
y = data.iloc[:, target_column].copy()


In [7]:
# Display the selected columns
print("\nFeatures used:")
print(X.columns)

print("\nTarget:")
print(data.columns[target_column])


Features used:
Index(['duration_ms', 'explicit', 'danceability', 'energy', 'loudness',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo'],
      dtype='object')

Target:
popularity


In [9]:
# Convert True/False values to 1/0 if necessary
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

X.iloc[:, 1] = le.fit_transform(X.iloc[:, 1])

print(X)

        duration_ms  explicit  danceability  energy  loudness  speechiness  \
0            230666         0         0.676  0.4610    -6.746       0.1430   
1            149610         0         0.420  0.1660   -17.235       0.0763   
2            210826         0         0.438  0.3590    -9.734       0.0557   
3            201933         0         0.266  0.0596   -18.515       0.0363   
4            198853         0         0.618  0.4430    -9.681       0.0526   
...             ...       ...           ...     ...       ...          ...   
113995       384999         0         0.172  0.2350   -16.393       0.0422   
113996       385000         0         0.174  0.1170   -18.318       0.0401   
113997       271466         0         0.629  0.3290   -10.895       0.0420   
113998       283893         0         0.587  0.5060   -10.889       0.0297   
113999       241826         0         0.526  0.4870   -10.204       0.0725   

        acousticness  instrumentalness  liveness  valence    te

C:\Users\anton\AppData\Local\Temp\ipykernel_1172\1920782536.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0 0 0 ... 0 0 0]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  X.iloc[:, 1] = le.fit_transform(X.iloc[:, 1])


In [13]:
#  Split into training and test sets
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0)

In [29]:
# ---------------------------------------------------------
# XGBoost Regression + GridSearchCV
# ---------------------------------------------------------

from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

# Create the XGBoost regression model
regressor = XGBRegressor(
    objective="reg:squarederror",
    random_state=0,
    n_jobs=1
)

# Define the parameters to test
param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5],
    "subsample": [0.8],
    "colsample_bytree": [0.8]
}

# Create GridSearchCV
grid_search = GridSearchCV(
    estimator=regressor,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=1
)

# Train all combinations
grid_search.fit(X_train, y_train)

# Best parameters
print("\nBest parameters:")
print(grid_search.best_params_)

# Best model
regressor = grid_search.best_estimator_

Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best parameters:
{'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}


In [30]:
# Predict popularity for the test set
# ---------------------------------------------------------

y_pred = regressor.predict(X_test)

In [31]:
# Evaluate the model
# ---------------------------------------------------------

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

print("\nModel performance")
print("-------------------------")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)


Model performance
-------------------------
MAE : 16.301525115966797
RMSE: 20.08904205271379
R²  : 0.18963950872421265


In [32]:
# ---------------------------------------------------------
# Find the best combination of factors
# ---------------------------------------------------------

n_candidates = 10000

candidate_indices = np.random.default_rng(0).choice(
    len(X),
    size=n_candidates,
    replace=True
)

candidate_songs = X.iloc[candidate_indices].copy()

# Predict popularity for candidate combinations
# ---------------------------------------------------------

candidate_predictions = regressor.predict(candidate_songs)

# Find the highest predicted popularity
# ---------------------------------------------------------

best_index = np.argmax(candidate_predictions)

best_combination = candidate_songs.iloc[best_index]

best_prediction = candidate_predictions[best_index]

# Display best combination
# ---------------------------------------------------------

print("\nBest predicted combination")
print("============================")

for feature in X.columns:
    print(
        f"{feature}: "
        f"{best_combination[feature]:.4f}"
    )

print("\nPredicted popularity:")
print(round(best_prediction, 2))

# Show the top 10 candidate combinations
# =========================================================

candidate_songs["predicted_popularity"] = candidate_predictions

top_candidates = candidate_songs.sort_values(
    "predicted_popularity",
    ascending=False
).head(10)

print("\nTop 10 predicted combinations:")
print(top_candidates)


Best predicted combination
duration_ms: 212360.0000
explicit: 1.0000
danceability: 0.9390
energy: 0.7420
loudness: -5.1710
speechiness: 0.0467
acousticness: 0.3210
instrumentalness: 0.0000
liveness: 0.1070
valence: 0.9240
tempo: 118.9780

Predicted popularity:
59.36

Top 10 predicted combinations:
        duration_ms  explicit  danceability   energy  loudness  speechiness  \
20413        212360         1         0.939  0.74200    -5.171       0.0467   
94070        165000         1         0.685  0.23000   -12.525       0.0388   
94723        134815         1         0.906  0.34900    -7.342       0.1060   
8201         283800         1         0.596  0.60500   -12.145       0.0255   
101263       182741         0         0.159  0.00979   -38.361       0.0409   
94028        121949         1         0.785  0.34300   -10.923       0.0862   
98152        115879         1         0.724  0.30500   -12.763       0.2570   
101505       199488         0         0.150  0.00550   -35.474      